# LogiScan: Unified Fallacy Classifier Training (Self-Contained)
This notebook handles the data preparation and training for the multi-head DeBERTa model.

**Required File:** `unified_training_data.json` (approx 3.5MB, 10.7K samples)

In [ ]:
import json
import os

import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from transformers import AutoConfig, AutoModel, AutoTokenizer, PreTrainedModel, Trainer, TrainingArguments

# Constants
DATA_PATH = "unified_training_data.json"
MODEL_NAME = "microsoft/deberta-v3-small"
OUTPUT_DIR = "logiscan_unified_model_v1.1"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Data Preparation

In [ ]:
if not os.path.exists(DATA_PATH):
    print(f"❌ ERROR: {DATA_PATH} not found!")
else:
    with open(DATA_PATH) as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    print(f"✅ Loaded {len(df)} samples.")

COARSE_MAP = {
    "ad_hominem": "Informal (Relevance)",
    "affirming_consequent": "Formal",
    "appeal_to_authority": "Informal (Relevance)",
    "appeal_to_emotion": "Informal (Relevance)",
    "appeal_to_nature": "Informal (Relevance)",
    "appeal_to_tradition": "Informal (Relevance)",
    "bandwagon": "Informal (Relevance)",
    "begging_the_question": "Informal (Presumption)",
    "composition": "Informal (Presumption)",
    "denying_antecedent": "Formal",
    "division": "Informal (Presumption)",
    "equivocation": "Informal (Ambiguity)",
    "false_cause": "Informal (Presumption)",
    "false_dilemma": "Informal (Presumption)",
    "hasty_generalization": "Informal (Presumption)",
    "moving_goalposts": "Informal (Relevance)",
    "no_true_scotsman": "Informal (Ambiguity)",
    "red_herring": "Informal (Relevance)",
    "slippery_slope": "Informal (Presumption)",
    "straw_man": "Informal (Relevance)",
    "tu_quoque": "Informal (Relevance)",
    "tu_quoque_contextual": "Informal (Relevance)",
    "factual_statement": "Non-Fallacious",
    "valid_reasoning": "Non-Fallacious"
}

df['coarse_label'] = df['fallacy'].map(lambda x: COARSE_MAP.get(x, "Informal (Other)"))

FINE_LABELS = sorted(df['fallacy'].unique().tolist())
COARSE_LABELS = sorted(df['coarse_label'].unique().tolist())
fine2id = {l: i for i, l in enumerate(FINE_LABELS)}
coarse2id = {l: i for i, l in enumerate(COARSE_LABELS)}
df['fine_id'] = df['fallacy'].map(fine2id)
df['coarse_id'] = df['coarse_label'].map(coarse2id)

## 2. Architecture Definitions

In [ ]:
class DebertaV3MultiHead(PreTrainedModel):
    config_class = AutoConfig
    base_model_prefix = "deberta"
    def __init__(self, config):
        super().__init__(config)
        self.num_labels_coarse = getattr(config, "num_labels_coarse", 5)
        self.num_labels_fine = getattr(config, "num_labels_fine", 24)
        self.deberta = AutoModel.from_config(config)
        self.pooler = nn.Linear(config.hidden_size, config.hidden_size)
        self.pooler_activation = nn.Tanh()
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.coarse_head = nn.Linear(config.hidden_size, self.num_labels_coarse)
        self.fine_head = nn.Linear(config.hidden_size, self.num_labels_fine)
        self.post_init()
    def forward(self, input_ids=None, attention_mask=None, labels_fine=None, labels_coarse=None, return_dict=True):
        outputs = self.deberta(input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        pooled = self.dropout(self.pooler_activation(self.pooler(cls_output)))
        c_logits, f_logits = self.coarse_head(pooled), self.fine_head(pooled)
        loss = None
        if labels_fine is not None and labels_coarse is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = 0.7 * loss_fct(f_logits.view(-1, self.num_labels_fine), labels_fine.view(-1)) + \
                   0.3 * loss_fct(c_logits.view(-1, self.num_labels_coarse), labels_coarse.view(-1))
        return {"loss": loss, "coarse_logits": c_logits, "fine_logits": f_logits} if return_dict else (loss, c_logits, f_logits)

class FallacyDataset(Dataset):
    def __init__(self, texts, fine_ids, coarse_ids, tokenizer):
        self.texts, self.fine_ids, self.coarse_ids, self.tokenizer = texts, fine_ids, coarse_ids, tokenizer
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        enc = self.tokenizer(str(self.texts[i]), padding='max_length', truncation=True, max_length=256, return_tensors='pt')
        return {'input_ids': enc['input_ids'].flatten(), 'attention_mask': enc['attention_mask'].flatten(),
                'labels_fine': torch.tensor(self.fine_ids[i]), 'labels_coarse': torch.tensor(self.coarse_ids[i])}

## 3. Training

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
config = AutoConfig.from_pretrained(MODEL_NAME)
config.update({"num_labels_coarse": len(COARSE_LABELS), "num_labels_fine": len(FINE_LABELS)})
model = DebertaV3MultiHead.from_pretrained(MODEL_NAME, config=config, ignore_mismatched_sizes=True)

X_train, X_val, yf_t, yf_v, yc_t, yc_v = train_test_split(df['text'].values, df['fine_id'].values, df['coarse_id'].values, test_size=0.1, random_state=42)
train_ds = FallacyDataset(X_train, yf_t, yc_t, tokenizer)
val_ds = FallacyDataset(X_val, yf_v, yc_v, tokenizer)

class MultiHeadTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels_fine, labels_coarse = inputs.pop("labels_fine"), inputs.pop("labels_coarse")
        outputs = model(**inputs, labels_fine=labels_fine, labels_coarse=labels_coarse)
        return (outputs["loss"], outputs) if return_outputs else outputs["loss"]

args = TrainingArguments(output_dir='./res', num_train_epochs=3, per_device_train_batch_size=16, evaluation_strategy="epoch", save_strategy="no", report_to="none")
trainer = MultiHeadTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds)
trainer.train()

## 4. Packaging and Download

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
with open(os.path.join(OUTPUT_DIR, "labels.json"), "w") as f: json.dump(FINE_LABELS, f)

print("✅ Model saved. Total size:")
!du -sh {OUTPUT_DIR}

# ONNX EXPORT
class ExportWrapper(torch.nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, input_ids, attention_mask):
        out = self.m(input_ids=input_ids, attention_mask=attention_mask)
        return out["coarse_logits"], out["fine_logits"]

model.to("cpu").eval()
dummy = tokenizer("test", return_tensors="pt", padding="max_length", truncation=True, max_length=256)
torch.onnx.export(ExportWrapper(model), (dummy["input_ids"], dummy["attention_mask"]), os.path.join(OUTPUT_DIR, "model.onnx"),
                  opset_version=14, input_names=["input_ids", "attention_mask"], output_names=["coarse", "fine"],
                  dynamic_axes={"input_ids": {0: "batch"}, "attention_mask": {0: "batch"}})

!zip -r logiscan_unified.zip {OUTPUT_DIR}
print("🚀 DOWNLOAD THIS FILE: logiscan_unified.zip")